# 05 · Choosing the Regularization Strength

Notebook 04 left one thing to the eye: *how much* to smooth. Picking $\lambda$
by eye is not reproducible. This notebook replaces the guess with three standard,
quantitative criteria — the **L-curve**, **ABIC**, and **cross-validation** —
all built into `geodef`.

## Learning objectives
- Understand the misfit–roughness trade-off as $\lambda$ varies.
- Locate a good $\lambda$ with the **L-curve** corner.
- Minimize the **Akaike Bayesian Information Criterion (ABIC)**.
- Choose $\lambda$ by **k-fold cross-validation**.
- Compare what the three criteria pick.

## The trade-off

As $\lambda$ grows, the data misfit $\lVert G\mathbf m - \mathbf d\rVert$ rises
(we fit the data less well) while the model roughness $\lVert L\mathbf m\rVert$
falls (the slip gets smoother). We want the sweet spot that fits the data to
about its noise level without inventing structure the data do not require.

- **L-curve.** Plot model norm vs. misfit on log–log axes as $\lambda$ sweeps.
  The curve is L-shaped; its **corner** is the best compromise — past it, misfit
  grows fast for little smoothing gain.
- **ABIC.** Treat regularization as a Bayesian *prior* and $\lambda$ as a
  hyperparameter. ABIC is (minus twice) the marginal likelihood; the
  $\lambda$ that **minimizes ABIC** is the most probable given the data.
- **Cross-validation.** Hold out some data, fit the rest, and predict the held-out
  points. The $\lambda$ that best predicts *unseen* data (minimum CV error)
  generalizes best.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# The higher-resolution megathrust from notebook 01, reused (copied) across
# notebooks 03-10 so every inversion targets the same "true" slip model.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: notebook 01's smooth Gaussian thrust asperity, dip-slip only.
# The model vector is blocked as [strike-slip (N) | dip-slip (N)], so the
# strike-slip half is zero.
i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.matrix(
    fault, geodef.GNSS(lon=glon, lat=glat, ve=_zero, vn=_zero, vu=_zero, se=_one, sn=_one, su=_one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.GNSS(
    lon=glon, lat=glat,
    ve=d_obs[0::3], vn=d_obs[1::3], vu=d_obs[2::3],
    se=np.full(n_sta, sigma), sn=np.full(n_sta, sigma), su=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")

40 patches, 36 stations, 108 observations


## The L-curve

Every $\lambda$ produces one (misfit, roughness) pair,

$$ \rho(\lambda) = \lVert G\hat{\mathbf m}_\lambda - \mathbf d\rVert,
   \qquad
   \eta(\lambda) = \lVert L\hat{\mathbf m}_\lambda\rVert . $$

Plotted on log-log axes as $\lambda$ sweeps, these trace an **L-shaped** curve.
The steep branch (small $\lambda$) is the *over-fitting* regime, where roughness
explodes while the misfit barely improves; the flat branch (large $\lambda$) is
the *over-smoothing* regime, where the misfit grows for no roughness gain. The
**corner** joining them is the best compromise, and `geodef` locates it at the
point of **maximum curvature**

$$ \lambda^\star = \arg\max_\lambda\;
   \kappa(\lambda) = \frac{\rho'\,\eta'' - \rho''\,\eta'}
   {\big((\rho')^2 + (\eta')^2\big)^{3/2}}, $$

with the derivatives taken along the log-log curve. `geodef.lcurve()` sweeps
$\lambda$, returns the $\rho$ and $\eta$ arrays plus the located corner, and
`.plot()` marks it.

In [2]:
lc = geodef.lcurve(fault, gnss, regularization="laplacian", components="dip",
                   regularization_range=(1e-2, 1e3), n=40)
lc.plot()
print(f"L-curve corner: lambda = {lc.optimal:.3g}")

L-curve corner: lambda = 3.67


## ABIC

The **Akaike Bayesian Information Criterion** treats the roughness penalty as a
Gaussian *prior* on the slip and the smoothing strength $\lambda$ as a
hyperparameter setting the prior variance. Integrating (marginalizing) the slip
$\mathbf m$ out of the posterior leaves a marginal likelihood for $\lambda$
alone; ABIC is $-2\log$ of it, so the **minimum** is the most probable strength.
For the linear-Gaussian problem (Fukuda & Johnson, 2008),

$$ \mathrm{ABIC}(\lambda) = N\,\log\!\Big(
   \lVert G\mathbf m_\lambda - \mathbf d\rVert_W^2
   + \lambda\,\lVert L\mathbf m_\lambda\rVert^2 \Big)
   \;-\; \log\big|\lambda\,L^{\mathsf T}L\big|_{+}
   \;+\; \log\big|G^{\mathsf T}WG + \lambda\,L^{\mathsf T}L\big|, $$

where $N$ is the number of data, $\lVert\mathbf r\rVert_W^2 =
\mathbf r^{\mathsf T} W\mathbf r$ is the noise-weighted misfit, and $|\cdot|_+$
is the product of the *positive* eigenvalues (a pseudo-determinant, since
$L^{\mathsf T}L$ is rank-deficient for a Laplacian). The first term rewards
fitting the data; the two log-determinants form an **Occam factor** that
penalizes an over-flexible prior, so ABIC trades data fit against model
complexity automatically. `geodef.abic_curve()` evaluates it across the sweep
(and `regularization_strength='abic'` selects the minimizer directly).

In [3]:
ab = geodef.abic_curve(fault, gnss, regularization="laplacian", components="dip",
                       regularization_range=(1e-2, 1e3), n=40)
ab.plot()
print(f"ABIC minimum:   lambda = {ab.optimal:.3g}")

ABIC minimum:   lambda = 1.13


## Cross-validation

$k$-fold **cross-validation** picks the $\lambda$ that best predicts data it
never saw. Partition the observations into $k$ disjoint folds. For each fold
$f$, fit the model on the *other* $k-1$ folds at strength $\lambda$ to get
$\hat{\mathbf m}^{(-f)}_\lambda$, then predict the held-out fold and accumulate
its squared error:

$$ \mathrm{CV}(\lambda) = \sum_{f=1}^{k}
   \big\lVert \mathbf d_f - G_f\,\hat{\mathbf m}^{(-f)}_\lambda \big\rVert^2 . $$

The $\lambda$ minimizing this out-of-sample error is chosen. Unlike the L-curve
and ABIC it needs no explicit prior or curvature heuristic -- only the data's
own predictive performance -- at the cost of $k$ inversions per trial
$\lambda$. Pass `regularization_strength='cv'` (with `cv_folds=k`) and `invert()`
runs the whole procedure internally, choosing the $\lambda$ that best predicts
held-out stations.

In [4]:
cv = geodef.invert.solve(fault, gnss, components="dip",
                   regularization="laplacian", regularization_strength="cv", cv_folds=5)
print(f"CV-selected:    lambda = {cv.regularization_strength:.3g}")

CV-selected:    lambda = 0.829


## Do they agree?

The three criteria are derived from different principles, so they need not land
on the same number — but they usually agree to within a factor of a few. Here are
all three choices and the slip each produces.

The top row is the slip each criterion produces (shared symmetric scale); the bottom row is each one's difference from the truth (recovered $-$ true). The residual maps are nearly identical, confirming the criteria land on very similar models.

In [5]:
choices = {
    "L-curve": lc.optimal,
    "ABIC": ab.optimal,
    "cross-val": cv.regularization_strength,
}
truth = slip_true[N:]
vmax = truth.max()
ncol = len(choices) + 1
fig, axes = plt.subplots(2, ncol, figsize=(3 * ncol, 6),
                         constrained_layout=True)

# Row 0: true + each method's slip, shared symmetric RdBu_r scale.
geodef.plot.slip(fault, truth, ax=axes[0, 0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, colorbar=False, title="True")
recovered = {}
for ax, (name, lam) in zip(axes[0, 1:], choices.items()):
    r = geodef.invert.solve(fault, gnss, components="dip",
                      regularization="laplacian", regularization_strength=lam)
    recovered[name] = r.slip_vector
    geodef.plot.slip(fault, r.slip_vector, ax=ax, cmap="RdBu_r",
                     vmin=-vmax, vmax=vmax, colorbar=False,
                     title=f"{name}\nlambda={lam:.2g}")

# Row 1: method - true, so the (small) differences between methods show.
diffs = {name: m - truth for name, m in recovered.items()}
dmax = max(np.abs(d).max() for d in diffs.values())
axes[1, 0].axis("off")
for ax, (name, d) in zip(axes[1, 1:], diffs.items()):
    geodef.plot.slip(fault, d, ax=ax, cmap="RdBu_r",
                     vmin=-dmax, vmax=dmax, colorbar=False,
                     title=f"{name} - true")

fig.colorbar(axes[0, 0].collections[-1], ax=axes[0, :], shrink=0.8,
             label="Slip (m)")
fig.colorbar(axes[1, 1].collections[-1], ax=axes[1, :], shrink=0.8,
             label="Recovered - true (m)")

## Exercises
1. **Compare.** Print the three $\lambda$ values side by side. How far apart are
   they? Does the recovered slip look meaningfully different?
2. **Noise sensitivity.** Increase `sigma` to `0.03` and re-run all three. Which
   criterion moves the most?
3. **Damping.** Repeat the L-curve with `regularization='damping'`. Is the corner as
   sharp?

## Checkpoint questions
1. Why is the corner of the L-curve a sensible compromise?
2. What is ABIC trying to maximize, in Bayesian terms?
3. What does cross-validation optimize that the L-curve and ABIC do not directly?

## Summary
- The misfit–roughness trade-off is controlled by $\lambda$.
- L-curve, ABIC, and CV each pick $\lambda$ by a different, reproducible rule.
- `geodef.lcurve`, `geodef.abic_curve`, and `regularization_strength='abic'|'cv'`
  automate all three.

**Next:** notebook 06 combines complementary datasets — GNSS and InSAR — in a
single joint inversion.